In [189]:
#imports the required libraries
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [190]:
#Defines common data path
data_path=r"D:\GIS-TU Dublin\Year_2\THESIS\DCC_Analysis"

In [191]:
#imports the Origin Destination Lines (Output from OD Cost Matrix Solver)
OD_gdf=gpd.read_file(data_path + r'\Data_used\OD_Output\ODLines_PTAL.shp')

In [192]:
OD_gdf.head()

,Name,OriginID,Destinatio,Destinat_1,Total_Leng,Total_Time,Shape_Leng,geometry
0,0 - 8230DB005130,1,2122,1,300.952560,5.011795,0.0,None
1,1 - 8230DB005130,2,2122,1,229.687253,3.823446,0.0,None
2,1 - 8230DB005131,2,2123,2,347.935070,5.797164,0.0,None
3,2 - 8230DB005131,3,2123,1,272.033780,4.526133,0.0,None
4,2 - 8230DB005130,3,2122,2,285.052883,4.745721,0.0,None


In [193]:
#Extracts stop_id
OD_gdf['stop_id']=OD_gdf['Name'].str.split('- ').str[1]
OD_gdf.head()


,Name,OriginID,Destinatio,Destinat_1,Total_Leng,Total_Time,Shape_Leng,geometry,stop_id
0,0 - 8230DB005130,1,2122,1,300.952560,5.011795,0.0,None,8230DB005130
1,1 - 8230DB005130,2,2122,1,229.687253,3.823446,0.0,None,8230DB005130
2,1 - 8230DB005131,2,2123,2,347.935070,5.797164,0.0,None,8230DB005131
3,2 - 8230DB005131,3,2123,1,272.033780,4.526133,0.0,None,8230DB005131
4,2 - 8230DB005130,3,2122,2,285.052883,4.745721,0.0,None,8230DB005130


In [194]:
#Filters the required columns only
OD_clean_gdf = OD_gdf[['Name','OriginID', 'Total_Leng','Total_Time', 'stop_id']]

In [195]:
OD_clean_gdf.head()

,Name,OriginID,Total_Leng,Total_Time,stop_id
0,0 - 8230DB005130,1,300.952560,5.011795,8230DB005130
1,1 - 8230DB005130,2,229.687253,3.823446,8230DB005130
2,1 - 8230DB005131,2,347.935070,5.797164,8230DB005131
3,2 - 8230DB005131,3,272.033780,4.526133,8230DB005131
4,2 - 8230DB005130,3,285.052883,4.745721,8230DB005130


In [196]:
#Reads the stops with frequency values
stops_freq=gpd.read_file(data_path + r'\GTFS_frequency_outputs\all_stops_freq_ITM.shp')

In [197]:
#Filters only required columns
stops_freq_clean=stops_freq[['stop_id','stop_name','route_type', 'freq_MP','freq_OP']]

In [198]:
stops_freq_clean.head()

,stop_id,stop_name,route_type,freq_MP,freq_OP
0,8220000002,Sheriff Street Upper,3,1,0
1,8220000005,Park West Road,3,2,1
2,8220000150,Abbey Street,3,3,2
3,8220000357,George's Quay,3,1,1
4,8220000372,East Wall Road,3,16,2


In [199]:
#Joins OD_Lines with stops
OD_gdf_freq = pd.merge(OD_clean_gdf, stops_freq_clean, on= 'stop_id',how='inner')
OD_gdf_freq.head(2)

,Name,OriginID,Total_Leng,Total_Time,stop_id,stop_name,route_type,freq_MP,freq_OP
0,0 - 8230DB005130,1,300.952560,5.011795,8230DB005130,Charleville Square,3,8,8
1,1 - 8230DB005130,2,229.687253,3.823446,8230DB005130,Charleville Square,3,8,8


In [200]:
#Calculates Accessibiliy Index (AI) for both peak and off-peak time following london connectivity assessment guidelines
def AI_calc_MP_OP(df, time_interval=60):
    """
    Calculate PTAL accessibility indices for both MP and OP frequencies from a single combined dataframe
    Returns the dataframe with AI_MP and AI_OP calculations
    """
    
    # Make a copy to avoid modifying the original
    result_df = df.copy()
    
    # Calculate for MP frequency
    result_df['time_interval_MP'] = time_interval
    result_df['SWT_MP'] = 0.5 * (result_df['time_interval_MP'] / result_df['freq_MP'].replace(0, 0.001))  # Avoid division by zero
    result_df['AWT_MP'] = result_df['SWT_MP'] + result_df['route_type'].apply(lambda x: 2 if x == 3 else 0.75)
    result_df['TAT_MP'] = result_df['Total_Time'] + result_df['AWT_MP']  # Using Total_Time as walk time
    result_df['EDF_MP'] = 0.5 * 60 / result_df['TAT_MP'].replace(0, 0.001)  # Avoid division by zero
    
    # Calculate for OP frequency
    result_df['time_interval_OP'] = time_interval
    result_df['SWT_OP'] = 0.5 * (result_df['time_interval_OP'] / result_df['freq_OP'].replace(0, 0.001))  # Avoid division by zero
    result_df['AWT_OP'] = result_df['SWT_OP'] + result_df['route_type'].apply(lambda x: 2 if x == 3 else 0.75)
    result_df['TAT_OP'] = result_df['Total_Time'] + result_df['AWT_OP']  # Using Total_Time as walk time
    result_df['EDF_OP'] = 0.5 * 60 / result_df['TAT_OP'].replace(0, 0.001)  # Avoid division by zero
    
    # Calculate AI_mode for both MP and OP
    def calc_ai_mode(series):
        largest = series.max()
        others_sum = series[series != largest].sum()
        return largest + 0.5 * others_sum
    
    # AI_mode for MP
    AI_mode_MP = result_df.groupby(['OriginID', 'route_type'])['EDF_MP'].apply(calc_ai_mode).reset_index(name='AI_mode_MP')
    result_df = result_df.merge(AI_mode_MP, on=['OriginID', 'route_type'], how='left')
    
    # AI_mode for OP
    AI_mode_OP = result_df.groupby(['OriginID', 'route_type'])['EDF_OP'].apply(calc_ai_mode).reset_index(name='AI_mode_OP')
    result_df = result_df.merge(AI_mode_OP, on=['OriginID', 'route_type'], how='left')
    
    # Calculate AI_total for both MP and OP
    AI_total_MP = result_df.groupby('OriginID')['AI_mode_MP'].sum().reset_index(name='AI_total_MP')
    AI_total_OP = result_df.groupby('OriginID')['AI_mode_OP'].sum().reset_index(name='AI_total_OP')
    
    # Merge AI_total back to the dataframe
    result_df = result_df.merge(AI_total_MP, on='OriginID', how='left')
    result_df = result_df.merge(AI_total_OP, on='OriginID', how='left')
    
    return result_df

# Usage with your dataframe:
OD_AI = AI_calc_MP_OP(OD_gdf_freq)

In [201]:
OD_AI.head()

,Name,OriginID,Total_Leng,Total_Time,stop_id,stop_name,route_type,freq_MP,freq_OP,time_interval_MP,SWT_MP,AWT_MP,TAT_MP,EDF_MP,time_interval_OP,SWT_OP,AWT_OP,TAT_OP,EDF_OP,AI_mode_MP,AI_mode_OP,AI_total_MP,AI_total_OP
0,0 - 8230DB005130,1,300.952560,5.011795,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.761795,2.787639,60,3.75,5.75,10.761795,2.787639,2.787639,2.787639,2.787639,2.787639
1,1 - 8230DB005130,2,229.687253,3.823446,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,9.573446,3.133668,60,3.75,5.75,9.573446,3.133668,4.558922,4.432688,9.117844,8.865376
2,2 - 8230DB005130,3,285.052883,4.745721,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.495721,2.858308,60,3.75,5.75,10.495721,2.858308,4.671203,4.348540,9.342407,8.697080
3,5 - 8230DB005130,6,304.031602,5.064989,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.814989,2.773928,60,3.75,5.75,10.814989,2.773928,2.773928,2.773928,2.773928,2.773928
4,6 - 8230DB005130,7,304.300025,5.061473,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.811473,2.774830,60,3.75,5.75,10.811473,2.774830,2.774830,2.774830,2.774830,2.774830


In [202]:
#Reads 100*100 Fishnet grid
fishnet_gdf=gpd.read_file(data_path + r'\Data_used\DCC\Grid_100_SA.shp')


In [203]:
fishnet_gdf.head()

,Id,geometry
0,0,"POLYGON ((713810.423 728481.740, 713810.423 72..."
1,0,"POLYGON ((713610.423 728581.740, 713610.423 72..."
2,0,"POLYGON ((713710.423 728581.740, 713710.423 72..."
3,0,"POLYGON ((713810.423 728581.740, 713810.423 72..."
4,0,"POLYGON ((713910.423 728581.740, 713910.423 72..."


In [204]:
#Reads Fishnet Grid Labels
fishnet_gdf_Point=gpd.read_file(data_path + r'\Data_used\DCC\Grid_100_label_SA.shp')

In [205]:
fishnet_gdf_Point.head()

,Id,geometry
0,0,POINT (713660.423 728531.740)
1,0,POINT (713760.423 728531.740)
2,0,POINT (713860.423 728531.740)
3,0,POINT (713960.423 728531.740)
4,0,POINT (714060.423 728531.740)


In [206]:
# Add an OriginID column to fishnet_gdf_Point matching OD_Lines
fishnet_gdf_Point = fishnet_gdf_Point.reset_index().rename(columns={"index":"OriginID"})


In [207]:
fishnet_gdf_Point.head()

,OriginID,Id,geometry
0,0,0,POINT (713660.423 728531.740)
1,1,0,POINT (713760.423 728531.740)
2,2,0,POINT (713860.423 728531.740)
3,3,0,POINT (713960.423 728531.740)
4,4,0,POINT (714060.423 728531.740)


In [208]:
#Merge Accessibility Index (AI) with 100*100 fishnet grid centroids
fishnet_points__with_AI = fishnet_gdf_Point.merge(OD_AI, on='OriginID', how='right')

In [209]:
fishnet_points__with_AI.head()

,OriginID,Id,geometry,Name,Total_Leng,Total_Time,stop_id,stop_name,route_type,freq_MP,freq_OP,time_interval_MP,SWT_MP,AWT_MP,TAT_MP,EDF_MP,time_interval_OP,SWT_OP,AWT_OP,TAT_OP,EDF_OP,AI_mode_MP,AI_mode_OP,AI_total_MP,AI_total_OP
0,1,0,POINT (713760.423 728531.740),0 - 8230DB005130,300.952560,5.011795,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.761795,2.787639,60,3.75,5.75,10.761795,2.787639,2.787639,2.787639,2.787639,2.787639
1,2,0,POINT (713860.423 728531.740),1 - 8230DB005130,229.687253,3.823446,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,9.573446,3.133668,60,3.75,5.75,9.573446,3.133668,4.558922,4.432688,9.117844,8.865376
2,3,0,POINT (713960.423 728531.740),2 - 8230DB005130,285.052883,4.745721,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.495721,2.858308,60,3.75,5.75,10.495721,2.858308,4.671203,4.348540,9.342407,8.697080
3,6,0,POINT (713660.423 728631.740),5 - 8230DB005130,304.031602,5.064989,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.814989,2.773928,60,3.75,5.75,10.814989,2.773928,2.773928,2.773928,2.773928,2.773928
4,7,0,POINT (713760.423 728631.740),6 - 8230DB005130,304.300025,5.061473,8230DB005130,Charleville Square,3,8,8,60,3.75,5.75,10.811473,2.774830,60,3.75,5.75,10.811473,2.774830,2.774830,2.774830,2.774830,2.774830


In [210]:
#Spatially joins AI with fishnet grid
fishnet_with_AI = gpd.sjoin(fishnet_gdf, fishnet_points__with_AI[['OriginID', 'AI_total_MP','AI_total_OP','geometry']], 
                            how='inner', predicate='contains')

fishnet_with_AI.head()


,Id,geometry,index_right,OriginID,AI_total_MP,AI_total_OP
0,0,"POLYGON ((713810.423 728481.740, 713810.423 72...",17,2,9.117844,8.865376
0,0,"POLYGON ((713810.423 728481.740, 713810.423 72...",1,2,9.117844,8.865376
1,0,"POLYGON ((713610.423 728581.740, 713610.423 72...",3,6,2.773928,2.773928
2,0,"POLYGON ((713710.423 728581.740, 713710.423 72...",4,7,2.774830,2.774830
3,0,"POLYGON ((713810.423 728581.740, 713810.423 72...",5,8,10.964070,10.614124


In [211]:
#Saves PTAL shapefile.
fishnet_with_AI.to_file(data_path + r"\Data_Extracted\PTAL\PTALs.shp", driver="ESRI Shapefile")

[211]:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
